# **CLIP-Dissect: описать канал словами, без разметки**

Практика к модулю [«Что сеть выучила: признаки и концепции»](https://open-xai-platform.web.app).

Network Dissection требует Broden — датасет с пиксельной разметкой понятий. Его надо где-то
взять, а найдёт он только те концепции, которые в нём есть. CLIP-Dissect снимает оба
ограничения: вместо масок — любой набор изображений, вместо словаря разметки — любой список слов.

Здесь мы соберём метод целиком:

- посчитаем матрицу похожести «картинка × слово» через CLIP;
- запишем отклики канала нашей сети на тот же набор изображений;
- сопоставим два ранжирования и получим текстовое описание канала — **без единой маски**.

In [ ]:
!pip install open_clip_torch -q

In [ ]:
import torch
import torch.nn.functional as F
import requests
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
import open_clip

torch.manual_seed(0);

## 1. Что нужно методу

Три вещи, и ни одна из них не требует разметки:

| Что | Зачем | Откуда берём здесь |
| --- | --- | --- |
| пробный набор изображений $D$ | на нём меряются и отклики канала, и похожесть на слова | картинки курса и их фрагменты |
| словарь понятий $S$ | из него выбирается описание | список слов, заданный руками |
| изучаемая сеть | её каналы мы и описываем | ResNet-50, слой `layer4` |

В настоящей работе $D$ берут в тысячи изображений, а словарь — в тысячи слов. Здесь набор
маленький, чтобы тетрадь считалась на CPU за минуту; логика от этого не меняется.

In [ ]:
def load(name):
    return Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/' + name).content)).convert('RGB')

sources = ['hog.jpg', 'cat.jpg', 'cat_and_dog.jpg', 'pig.png']
originals = [load(n) for n in sources]

# Пробный набор: сами картинки и по четыре фрагмента из каждой — так откликов больше,
# а разметка по-прежнему не нужна.
probe = []
for im in originals:
    w, h = im.size
    probe.append(im)
    for box in [(0, 0, w // 2, h // 2), (w // 2, 0, w, h // 2),
                (0, h // 2, w // 2, h), (w // 2, h // 2, w, h)]:
        probe.append(im.crop(box))
print('изображений в пробном наборе:', len(probe))

In [ ]:
vocabulary = [
    'a photo of a pig', 'a photo of a cat', 'a photo of a dog',
    'fur texture', 'grass', 'a snout', 'an eye', 'a paw',
    'a wooden fence', 'sky', 'dirt and mud', 'whiskers',
]
print('слов в словаре:', len(vocabulary))

## 2. Матрица похожести «картинка × слово»

CLIP кладёт изображения и тексты в одно пространство, поэтому похожесть — это косинус между
эмбеддингами. Нашей сети здесь ещё нет: это свойство самого CLIP.

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

with torch.no_grad():
    img_emb = clip_model.encode_image(torch.stack([clip_preprocess(im) for im in probe]))
    txt_emb = clip_model.encode_text(tokenizer(vocabulary))
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)

similarity = img_emb @ txt_emb.T           # (изображения, слова)
print('матрица похожести:', tuple(similarity.shape))

## 3. Отклики канала изучаемой сети

Для каждого изображения пробного набора записываем среднюю активацию канала. Получается
вектор длины «число изображений» — по одному на канал.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

store = {}
handle = model.layer4[-1].register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
with torch.no_grad():
    model(torch.stack([transform(im) for im in probe]))
handle.remove()

activations = store['a'].mean(dim=(2, 3))   # (изображения, каналы)
print('матрица активаций:', tuple(activations.shape))

## 4. Сопоставление двух ранжирований

Наивно взять корреляцию — но, как отмечают авторы, канал часто отзывается сильно лишь на малую
долю изображений, и корреляция такой профиль размывает. Поэтому мера намеренно асимметрична:
важно, что канал возбуждают именно «полосатые» картинки, а не то, что все полосатые его возбуждают.

Возьмём простую версию той же идеи: смотрим только на верхушку активаций канала и спрашиваем,
какое слово в среднем ближе всего к этим изображениям.

In [ ]:
def describe(channel, top_k=5):
    """Слово словаря, лучше всего описывающее верхушку активаций канала."""
    top = torch.topk(activations[:, channel], top_k).indices
    score = similarity[top].mean(0)
    best = score.argmax().item()
    return vocabulary[best], score[best].item()

for channel in (0, 7, 42, 100, 512, 1000, 2047):
    word, score = describe(channel)
    print(f'канал {channel:4d} → «{word}»  (похожесть {score:.3f})')

**Задание 1.** Добавьте в словарь пять своих слов — например, `'a wheel'`, `'a window'`,
`'water'`. Изменились ли описания каналов? Что это говорит о зависимости метода от словаря?

**Задание 2.** Найдите канал, у которого верхушка активаций состоит из фрагментов разных
изображений с разными объектами. Какое описание он получит и почему по нему нельзя судить
о канале?

In [ ]:
# Ваш код здесь

## 5. Чего метод не даёт

Проверим главное ограничение из урока своими руками: **CLIP-Dissect приписывает каналу одно
понятие**. Если канал полисемантичен, второе и третье понятия просто не будут названы.

In [ ]:
channel = 42
top = torch.topk(activations[:, channel], 5).indices
score = similarity[top].mean(0)
order = torch.topk(score, 3)

print(f'канал {channel} — три ближайших слова, а не одно:')
for v, i in zip(order.values, order.indices):
    print(f'   «{vocabulary[i]}»  {v:.3f}')
print('\nРазрыв между первым и вторым:', round((order.values[0] - order.values[1]).item(), 3))

**Задание 3.** Если разрыв между первым и вторым словом мал, описание канала одним словом
теряет половину картины. Посчитайте этот разрыв для десяти произвольных каналов. У скольких
из них он меньше 0.01?

**Задание 4.** Метод описывает канал словами **чужой модели** — CLIP. Придумайте, как проверить,
что описание говорит о нашей сети, а не о слепых пятнах CLIP. (Подсказка из урока: показать
топ-активирующие изображения человеку.)

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Разметка не нужна.** Метод сравнивает не области на картинке, а два ранжирования
  изображений — «на что сильнее реагирует канал» против «что больше похоже на слово».
  Ни масок, ни порога $T_k$ здесь нет вовсе.
- **Словарь можно менять свободно** — и от него зависит ответ. Это гиперпараметр объяснения,
  и его надо указывать вместе с результатом.
- **Одно понятие на канал.** Словарную проблему Network Dissection метод снимает,
  а проблему «одна единица — одна концепция» — нет.
- **Описание порождает другая модель.** Ошибки и предвзятости CLIP попадают прямо в наше
  объяснение, и проверить это можно только внешне.

**Метки:** глобальный, model-agnostic по отношению к изучаемой сети — нужны только активации.
Вход: сеть, любой набор изображений, любой словарь. Выход: текстовое описание канала.